In [ ]:
%load_ext autoreload
%autoreload 2

from tabulate import tabulate
import torch.nn as nn

from fart.constants import MAGNITUDE
from fart.model.evaluate_model import evaluate_model
from fart.model.train_model import train_model
from fart.model.prepare_datasets import prepare_datasets
from fart.utils import get_data_filepath, get_project_root
from fart.visualization.evaluation_line_chart import evaluation_line_chart
from fart.visualization.plot_styles import apply_plot_styles

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market = "BTC-EUR"
interval = "1d"
data_filepath = get_data_filepath(assets_dir, market, interval)

batch_size = 16
learning_rate = 0.001
num_epochs = 500
num_lags = 100
num_neurons_in_hidden_layers = 20
train_size = 0.8

In [ ]:
x_train, y_train, x_test, y_test = prepare_datasets(
    data_filepath=data_filepath,
    target=MAGNITUDE,
    num_lags=num_lags,
    train_size=train_size,
)

In [ ]:
def build_model_fn():
    return nn.Sequential(
        nn.Linear(num_lags, num_neurons_in_hidden_layers),
        nn.ReLU(),
        nn.Linear(num_neurons_in_hidden_layers, num_neurons_in_hidden_layers),
        nn.ReLU(),
        nn.Linear(num_neurons_in_hidden_layers, 1),
    )

In [ ]:
n_splits = 5
model, cv_results = train_model(
    build_model_fn=build_model_fn,
    x_train=x_train,
    y_train=y_train,
    batch_size=batch_size,
    learning_rate=learning_rate,
    num_epochs=num_epochs,
    n_splits=n_splits,
)

In [ ]:
(
    y_train_pred,
    y_test_pred,
    accuracy_train,
    accuracy_test,
    rmse_train,
    rmse_test,
    mae_train,
    mae_test,
) = evaluate_model(
    model=model,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
)

print(
    tabulate(
        [
            [
                "Train",
                round(accuracy_train, 2),
                round(rmse_train, 6),
                round(mae_train, 6),
            ],
            [
                "Test",
                round(accuracy_test, 2),
                round(rmse_test, 6),
                round(mae_test, 6),
            ],
        ],
        headers=["Dataset", "Accuracy", "RMSE", "MAE"],
    )
)

In [ ]:
evaluation_line_chart(
    y_train=y_train,
    y_test=y_test,
    y_pred=y_test_pred,
)

In [ ]:
evaluation_line_chart(
    y_test=y_test,
    y_pred=y_test_pred,
)